<a href="https://colab.research.google.com/github/accoelhos/Trabalho_Teoria_Computacao/blob/main/trabalho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Definição do Problema

## 1.1 Informações Importantes nos Estados

Os estados do autômato precisam rastrear


*   Contexto base: se ele está lendo algo decimal (0-9), haxadecimal (A-F) ou científico.
*   Estágio da parte numérica: se o dígito que ele leu pertence à parte principal do número, ao expoente (em caso de científico) ou à parte imaginária (em caso de número complexo).
* Flags de símbolos: deve ter conhecimento de prefixos como 0x e #, sufixos como h, i e separadores como ., e, E.

## 1.2 Representação como linguagem L

A linguagem do autômato é a união de cinco sublinguagens

$$L = L_{int} \cup L_{real} \cup L_{sci} \cup L_{hex} \cup L_{comp}$$

Com o alfabeto $\Sigma = \{0..9, A..F, a..f, ., +, -, e, E, x, h, i, \#\}$.

## 1.3 Estratégia para solução do problema

1. Desenvolver os autômatos para cada umas das notações possíveis separadamente

2. Fazer o produto de todos esses autômatos levando em consideração casos de duplicidade de significado como o de E/e para notação científica e hexadecimal e o 0 para sufixo de hexadecimal e como número.

3. Implementar no código para fazer testes automatizados

4. Em caso de erro, tomar esses passos em ordem e caso o primeiro resolva não é necessário seguir adiante:
  - Verificar se há erros no código
  - Verificar se há erros no autômato produto
  - Verificar se há erros no autômato da notação que falhou no teste

# 2. Definição Formal do AF

## 2.1 Descrição dos estados ($Q$)

| Estado | Propósito | Categoria de Aceitação |
| :--- | :--- | :--- |
| **q0** | **Estado Inicial**: Ponto de partida da leitura. | - |
| **q1** | **Sinal**: Após leitura de sinal inicial (+ ou -). | - |
| **q2** | **Inteiro**: Lendo dígitos decimais. | **Inteiros** |
| **q3** | **Ponto Isolado**: Após leitura de "." inicial (ex: .5). | - |
| **q4** | **Real**: Lendo dígitos da parte fracionária. | **Reais** |
| **q5** | **Expoente**: Após o caractere de notação científica (e, E). | - |
| **q6** | **Sinal do Expoente**: Após + ou - dentro da parte científica. | - |
| **q7** | **Científico**: Lendo dígitos do expoente. | **Notação Científica** |
| **q8** | **Prefixo Hexa**: Após detectar "0x", "0X" ou "#". | - |
| **q9** | **Hexadecimal**: Lendo dígitos de base 16 (0-9, A-F). | **Hexadecimal** |
| **q10** | **Complexo**: Após validar o sufixo "i" da parte imaginária. | **Complexos** |
| **qerr** | **Erro**: Estado de rejeição para sequências inválidas. | - |

## 2.2 Alfabeto de entrada ($\Sigma$)

$$\Sigma = \{0..9, A..F, a..f, ., +, -, e, E, x, h, i, \#\}$$

### 2.3 Descrição da Função de Transição ($\delta$)
Abaixo, a tabela de transição que mapeia o comportamento do autômato.

*Foram utilizadas as seguintes convenções:*

- `n` → dígitos `1–9`
- `0` → dígito `0`
- `s` → `{+, -}`
- `e` → `{e, E}`
- `l` → `{A–F, a–f}`
- `i` → `i`
- `h` → `h`

| Estado Atual | Símbolo (σ) | Próximo Estado | Descrição |
|--------------|------------|----------------|----------|
| **q0** | n | q1 | Início com dígito 1–9 |
| **q0** | 0 | q3 | Início com zero |
| **q0** | s | q2 | Sinal inicial |
| **q0** | . | q4 | Número começando com ponto |
| **q0** | # | q13 | Prefixo hexadecimal (#) |
| **q0** | l | q14 | Início direto como hexadecimal |
| **q0** | e | q14 | Letra tratada como hexadecimal |
| **q1** | n, 0 | q1 | Loop da parte inteira |
| **q1** | . | q5 | Parte decimal |
| **q1** | e | q16 | Início do expoente |
| **q1** | s | q6 | Possível número complexo |
| **q1** | h | q15 | Sufixo hexadecimal |
| **q2** | n, 0 | q11 | Dígitos após sinal |
| **q3** | n, 0 | q1 | Continua número |
| **q3** | x | q13 | Prefixo 0x |
| **q3** | . | q5 | Vai para float |
| **q3** | l, e | q14 | Hexadecimal |
| **q4** | n, 0 | q5 | Parte fracionária |
| **q5** | n, 0 | q5 | Loop decimal |
| **q5** | e | q20 | Expoente |
| **q5** | s | q6 | Complexo |
| **q6** | n, 0 | q7 | Parte imaginária |
| **q7** | n, 0 | q8 | Continua número |
| **q7** | . | q9 | Parte decimal do imaginário |
| **q7** | i | q10 | Final complexo |
| **q8** | n, 0 | q8 | Loop inteiro |
| **q8** | . | q21 | Decimal |
| **q8** | i | q10 | Final complexo |
| **q9** | n, 0 | q21 | Continua decimal |
| **q11** | n, 0 | q11 | Loop inteiro |
| **q11** | . | q12 | Decimal |
| **q11** | e | q20 | Científico |
| **q12** | n, 0 | q12 | Loop decimal |
| **q13** | n, 0, l, e | q14 | Corpo hexadecimal |
| **q14** | n, 0, l, e | q14 | Loop hexadecimal |
| **q14** | h | q15 | Final hexadecimal |
| **q16** | n, 0 | q17 | Expoente direto |
| **q16** | s | q18 | Sinal do expoente |
| **q16** | l, e | q14 | Hexadecimal |
| **q17** | n, 0 | q17 | Loop expoente |
| **q17** | l, e | q14 | Hexadecimal |
| **q17** | h | q15 | Final hexadecimal |
| **q18** | n, 0 | q19 | Expoente após sinal |
| **q19** | n, 0 | q19 | Loop expoente |
| **q20** | n, 0 | q19 | Expoente |
| **q20** | s | q18 | Permite Sinal após euler |
| **q21** | n, 0, i | q10 | Final complexo |

### 2.4 Estados Finais ($F$)
O conjunto de estados de aceitação é definido por:
$$F = \{q_2, q_4, q_7, q_9, q_{10}\}$$

# 3. Implementação

- Código completo da classe ```FiniteAutomata```
- Funções auxiliares



In [21]:
from typing import Dict, Tuple, Optional

#Definição do alfabeto
#Obs.: Apenas o num, sinal, euler e letra são utilizados no código, os outros são para visibilidade do usuário
# 0 não está em num e nem E/e em letra pois são considerados como símbolos a parte já que podem ser ambíguos
num = [ "1", "2", "3", "4", "5", "6", "7", "8", "9"]
sinal = ["-", "+"]
ponto = ['.']
img = ['i']
euler = ['E', 'e']
letra = ['A', 'B', 'C', 'D', 'F', 'a', 'b', 'c', 'd', 'f']
prefixo1 = ['0']
prefixo2 = ['x']
prefixo3 = ['#']
sufixo = ['h']

class FiniteAutomata:
  delta: Dict[Tuple[str, str], str] = None

  def __init__(self, tape):
    # Inicialização

    #Tape é a fita que será analisada
    self.tape = tape

    #Head é a posição do caractere na fita analisada
    self.head = 0

    #Estado_atual para validar com as funções de transição e estado final
    self.estado_atual = 'q0'

    #Definição das funções de transição
    self.delta = {
        ('q0', 'n'): 'q1',
        ('q0', 's'): 'q2',
        ('q0', '0'): 'q3',
        ('q0', '.'): 'q4',
        ('q0', '#'): 'q13',
        ('q0', 'l'): 'q14',
        ('q0', 'e'): 'q14',

        ('q1', 'n'): 'q1',
        ('q1', '0'): 'q1',
        ('q1', '.'): 'q5',
        ('q1', 'e'): 'q16',
        ('q1', 's'): 'q6',
        ('q1', 'h'): 'q15',

        ('q2', 'n'): 'q11',
        ('q2', '0'): 'q11',

        ('q3', 'n'): 'q1',
        ('q3', '0'): 'q1',
        ('q3', 'x'): 'q13',
        ('q3', '.'): 'q5',
        ('q3', 'l'): 'q14',
        ('q3', 'e'): 'q14',

        ('q4', 'n'): 'q5',
        ('q4', '0'): 'q5',

        ('q5', 'n'): 'q5',
        ('q5', '0'): 'q5',
        ('q5', 'e'): 'q20',
        ('q5', 's'): 'q6',

        ('q6', 'n'): 'q7',
        ('q6', '0'): 'q7',

        ('q7', 'n'): 'q8',
        ('q7', '0'): 'q8',
        ('q7', '.'): 'q9',
        ('q7', 'i'): 'q10',

        ('q8', 'n'): 'q8',
        ('q8', '0'): 'q8',
        ('q8', '.'): 'q21',
        ('q8', 'i'): 'q10',

        ('q9', 'n'): 'q21',
        ('q9', '0'): 'q21',

        ('q11', 'n'): 'q11',
        ('q11', '0'): 'q11',
        ('q11', '.'): 'q12',
        ('q11', 'e'): 'q20',

        ('q12', 'n'): 'q12',
        ('q12', '0'): 'q12',

        ('q13', 'n'): 'q14',
        ('q13', '0'): 'q14',
        ('q13', 'l'): 'q14',
        ('q13', 'e'): 'q14',

        ('q14', 'n'): 'q14',
        ('q14', '0'): 'q14',
        ('q14', 'l'): 'q14',
        ('q14', 'e'): 'q14',
        ('q14', 'h'): 'q15',

        ('q16', 'n'): 'q17',
        ('q16', '0'): 'q17',
        ('q16', 'l'): 'q14',
        ('q16', 'e'): 'q14',
        ('q16', 's'): 'q18',

        ('q17', '0'): 'q17',
        ('q17', 'n'): 'q17',
        ('q17', 'l'): 'q14',
        ('q17', 'e'): 'q14',
        ('q17', 'h'): 'q15',

        ('q18', '0'): 'q19',
        ('q18', 'n'): 'q19',

        ('q19', '0'): 'q19',
        ('q19', 'n'): 'q19',

        ('q20', 'n'): 'q19',
        ('q20', '0'): 'q19',
        ('q20', 's'): 'q18',

        ('q21', 'n'): 'q10',
        ('q21', '0'): 'q10',
        ('q21', 'i'): 'q10',
    }

    #Definição dos estados finais e do estado de erro
    self.estado_final = {'q1', 'q3', 'q5', 'q10', 'q11', 'q12', 'q14', 'q15', 'q16', 'q17', 'q19'}
    self.estado_erro   = 'qerr'

  def step(self):
      """
      Executa um passo do AF
      Retorna: keep_running
      """

      try:
        self.tape = str(self.tape)
      except E:
        print('A sequencia não pode ser convertida em string')
        self.estado_atual = 'qerr'
        return False

      #Unifica símbolos que pertencem ao mesmo significado
      simbolo = self.tape[self.head]
      if simbolo in num:
        simbolo = 'n'
      if simbolo in letra:
        simbolo = 'l'
      if simbolo in euler:
        simbolo = 'e'
      if simbolo in sinal:
        simbolo = 's'
      transicao = (self.estado_atual, simbolo)

      #Verifica se o estado com o símbolo são válidos, se não é erro e retorna false
      self.estado_atual = self.delta.get(transicao, self.estado_erro)
      self.head += 1

      keep_running = (self.head < len(self.tape) and self.estado_atual != self.estado_erro)
      return keep_running

  def execute(self):
      """
      Executa o AF para a fita fornecida
      Retorna: (accept)
      """

      #Verifica a fita caractere por caractere até o final e então verifica se está no estado final, se estiver é True, se não False
      if len(self.tape) <= 0:
        return False

      while self.step():
        pass

      accept = self.estado_atual in self.estado_final

      return accept

# 4. Testes

- Casos pequenos
- Verificação de correção


In [22]:
# Testes unitários por notação, válidos e inválidos
casos = [
    #Inteiros

    #--Válidos
    ("123",       True),
    ("-456",      True),
    ("+789",      True),
    ("0",         True),

    #--Invalidos
    ("",          False),   # fita vazia
    ("+",         False),   # só sinal, sem dígito
    ("--5",       False),   # dois sinais
    ("3+",       False),
    ("01a", False),
    ("++1", False),
    ("-+1", False),
    ("1-", False),

    # Reais

    #--Válidos
    ("0.", True),
    (".0", True),
    ("000.000", True),
    ("3.14",      True),
    ("-0.5",      True),
    ("+2.0",      True),
    (".5",        True),
    ("5.",        True),

    #--Invalidos
    (".", False),
    ("..5", False),
    ("5..0", False),
    ("5.0.1", False),
    (".e1", False),

    #Notação Científica

    #--Válidos
    ("1.23e4",    True),
    ("-5E-10",    True),
    ("6e+7",      True),
    (".5e3",      True),
    ("2.e-2",     True),
    ("1e0", True),
    ("1E+0", True),
    ("1e-0", True),
    ("0e0", True),

    #--Inválidos
    ("1e+", False),
    ("1e-", False),
    ("1e1.5", False),      # expoente não pode ser float

    #Hexadecimal

    #--Válidos
    ("0xFF",      True),
    ("0x1A3",     True),
    ("#FF",       True),
    ("FFh",       True),
    ("0x0", True),
    ("0xabc", True),
    ("#1A", True),
    ("ABCDEFh", True),
    ("e10", True),        # sem base
    ("1e", True),         # sem expoente

    #--Inválidos
    ("0x", False),
    ("#", False),
    ("0xG", False),        # G não é hex
    ("0x1.2", False),      # ponto inválido
    ("h", False),
    ("FFhh", False),

    #Complexos

    #--Válidos
    ("3+4i",      True),
    ("2.5-1.3i",  True),

    #--Inválidos
    ("i", False),          # você NÃO definiu imaginário puro
    ("3+i", False),        # falta número após +
    ("3+4", False),        # sem i
    ("3+4ii", False),
    ("3++4i", False),
    ("3+4i5", False),

    #Casos misturados
    ("000e10", True),
    (".0e0", True),
    ("0.e1", True),

    ("0x1e", True),
    ("0x1e+2", False),

    ("3+4e2i", False),
    ("3+4.0e2i", False),

    ("1.2.3", False),
    ("0x1e10", True),
    ("123h", True),
    ("123he", False),
    ("3.14i", False),

]

for fita, esperado in casos:
    af      = FiniteAutomata(fita)
    result  = af.execute()
    status  = "Acertou" if result == esperado else "Errou"
    print(f"{status}  '{fita!s:<8}' → {'aceite' if result else 'rejeita'}  (esperado: {'aceite' if esperado else 'rejeita'})")

Acertou  '123     ' → aceite  (esperado: aceite)
Acertou  '-456    ' → aceite  (esperado: aceite)
Acertou  '+789    ' → aceite  (esperado: aceite)
Acertou  '0       ' → aceite  (esperado: aceite)
Acertou  '        ' → rejeita  (esperado: rejeita)
Acertou  '+       ' → rejeita  (esperado: rejeita)
Acertou  '--5     ' → rejeita  (esperado: rejeita)
Acertou  '3+      ' → rejeita  (esperado: rejeita)
Acertou  '01a     ' → rejeita  (esperado: rejeita)
Acertou  '++1     ' → rejeita  (esperado: rejeita)
Acertou  '-+1     ' → rejeita  (esperado: rejeita)
Acertou  '1-      ' → rejeita  (esperado: rejeita)
Acertou  '0.      ' → aceite  (esperado: aceite)
Acertou  '.0      ' → aceite  (esperado: aceite)
Acertou  '000.000 ' → aceite  (esperado: aceite)
Acertou  '3.14    ' → aceite  (esperado: aceite)
Acertou  '-0.5    ' → aceite  (esperado: aceite)
Acertou  '+2.0    ' → aceite  (esperado: aceite)
Acertou  '.5      ' → aceite  (esperado: aceite)
Acertou  '5.      ' → aceite  (esperado: aceite)
Acer

# 5. Demonstração Formal

- Prova de corretude do AF apresentado

# 6. Conclusão

- Principais aprendizados
- Limitações do estudo

## Principais aprendizados


1. A mistura de 0 como prefixo e número e o E/e como euler e letra de hexadecimal foram os fatores que mais dificultaram o desenvolvimento do autômato

2. Após desenvolver o código principal, trocar o autômato no self.delta foi fácil, possibilitando correções necessárias do processo

## Limitações do Estudo

- O autômato apenas identifica se é aceitou ou não, não retorna de que notação aquela sequência pertence
- Para autômatos muito grande o código começa a ficar menos legível por conta do grande mapeamento de transições